# 03 — Tier-1 MCTS-HNM Benchmarks: Multi-Seed Baselines & Hierarchy Ablations

**Feature**: `F4` (`04-tier1-benchmark-suite` | `EPIC-04`)  
**Architecture Decisions**: `AD-001` (Tier-1 Embedding Prototyping), `AD-002` (Semantic vs. Ward Confusion Tree), `AD-004` (Scenario 1 & 2 Regimes), `AD-005` (Strict GiantHunter vs. Enhanced MCTS-HNM), `AD-008` (Remote JupyterHub Execution)

This notebook orchestrates the core experimental axes across $N = 5$ random seeds:
1. **Axis 1 (Negative Samplers Comparison)**: Uniform Sampling vs. OHEM vs. Strict GiantHunter vs. Enhanced MCTS-HNM in Scenario 1 (One-vs-Rest, $\rho = 1:50$).
2. **Axis 2 (Hierarchy Geometry Ablation)**: Native Semantic Tree vs. Data-Driven Ward Confusion Tree under Enhanced MCTS-HNM.
3. **Axis 3 (Hyperparameter Sensitivity)**: Exploration coefficient $c \in \{0.5, 1.0, 2.0, 4.0\}$ and momentum $\alpha \in \{0.1, 0.3, 0.5, 0.7\}$.
4. **Axis 4 (Scenario 2 Curriculum)**: 100-way Multi-Class evaluation.
5. **Statistical Significance**: Paired Student's $t$-test and Wilcoxon signed-rank tests ($p$-values) and export to `results/`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.data.cifar100_taxonomy import get_cifar100_taxonomy
from src.data.extract_embeddings import load_embedding_cache
from src.training.benchmark_runner import run_benchmark
from src.utils.seed import set_global_seed

SEED = set_global_seed(42)
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {REPO_ROOT} | Results directory: {RESULTS_DIR}")

## 1. Load Precomputed ResNet-18 Embeddings

Loads `embeddings/cifar100_resnet18.pt` generated by Notebook 01, or synthesizes a deterministic fixture if cache is absent.

In [ ]:
CACHE_PATH = REPO_ROOT / "embeddings" / "cifar100_resnet18.pt"

if CACHE_PATH.exists():
    bundle = load_embedding_cache(CACHE_PATH)
    train_features = bundle["train"]["embeddings"]
    train_labels = bundle["train"]["fine_labels"]
    val_features = bundle["val"]["embeddings"]
    val_labels = bundle["val"]["fine_labels"]
    test_features = bundle["test"]["embeddings"]
    test_labels = bundle["test"]["fine_labels"]
    print(f"Loaded cached embeddings: Train={tuple(train_features.shape)}, Val={tuple(val_features.shape)}, Test={tuple(test_features.shape)}")
else:
    rng = np.random.default_rng(SEED)
    train_labels = np.repeat(np.arange(100, dtype=np.int64), 30)
    val_labels = np.repeat(np.arange(100, dtype=np.int64), 10)
    test_labels = np.repeat(np.arange(100, dtype=np.int64), 10)
    prototypes = rng.normal(size=(100, 64)).astype(np.float32)
    train_features = torch.from_numpy(prototypes[train_labels] + rng.normal(scale=0.8, size=(train_labels.size, 64)).astype(np.float32))
    val_features = torch.from_numpy(prototypes[val_labels] + rng.normal(scale=0.8, size=(val_labels.size, 64)).astype(np.float32))
    test_features = torch.from_numpy(prototypes[test_labels] + rng.normal(scale=0.8, size=(test_labels.size, 64)).astype(np.float32))
    print("Cache not found; initialized deterministic synthetic embeddings fixture for smoke execution.")

## 2. Axis 1: Sampler Comparison in Scenario 1 (Uniform vs. OHEM vs. Strict vs. Enhanced MCTS)

Compares the four negative sampling strategies across multiple seeds on Scenario 1 (One-vs-Rest, target class `0` / `beaver`, imbalanced positive pool).

In [ ]:
SEEDS = (42, 43, 44, 45, 46)

res_axis1 = run_benchmark(
    scenario="scenario1",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("uniform", "ohem", "mcts_strict", "mcts_enhanced"),
    tree_types=("semantic",),
    seeds=SEEDS,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_axis1_samplers",
    num_steps=100,
    eval_every=25,
    target_class_scenario1=0,
    val_features=val_features,
    val_labels=val_labels,
)

df_axis1 = pd.read_csv(RESULTS_DIR / "tier1_axis1_samplers.csv")
summary_axis1 = df_axis1.groupby("strategy")[["auprc", "f1", "fpr_at_95_recall"]].agg(["mean", "std"])
print("=== Axis 1: Sampler Comparison Results (Scenario 1) ===")
print(summary_axis1)
print("\nStatistical tests against Enhanced MCTS-HNM:")
for comp, stats in res_axis1["statistical_comparisons"].items():
    print(f"  {comp}: diff={stats['mean_diff']:.4f}, t_p={stats['t_pvalue']:.4e}, w_p={stats['wilcoxon_pvalue']:.4e}")

## 3. Axis 2: Hierarchy Geometry Ablation (Semantic vs. Ward Confusion Tree)

Tests whether empirical visual error clustering (Ward link on validation confusion matrix) improves MCTS mining efficiency over static taxonomy.

In [ ]:
res_axis2 = run_benchmark(
    scenario="scenario1",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("mcts_enhanced",),
    tree_types=("semantic", "confusion_ward"),
    seeds=SEEDS,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_axis2_tree_ablation",
    num_steps=100,
    eval_every=25,
    target_class_scenario1=0,
    val_features=val_features,
    val_labels=val_labels,
)

df_axis2 = pd.read_csv(RESULTS_DIR / "tier1_axis2_tree_ablation.csv")
summary_axis2 = df_axis2.groupby("tree_type")[["auprc", "f1", "fpr_at_95_recall"]].agg(["mean", "std"])
print("=== Axis 2: Tree Geometry Ablation Results ===")
print(summary_axis2)

## 4. Axis 3: Scenario 2 Multi-Class Curriculum Benchmark

Evaluates 100-way multi-class classification accuracy, macro-F1, and worst-10-class recall under MCTS curriculum.

In [ ]:
res_s2 = run_benchmark(
    scenario="scenario2",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("uniform", "mcts_enhanced"),
    tree_types=("semantic",),
    seeds=(42, 43),
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_scenario2_curriculum",
    num_steps=60,
    eval_every=20,
    val_features=val_features,
    val_labels=val_labels,
)

df_s2 = pd.read_csv(RESULTS_DIR / "tier1_scenario2_curriculum.csv")
summary_s2 = df_s2.groupby("strategy")[["top1_accuracy", "macro_f1", "worst_10_class_recall"]].agg(["mean", "std"])
print("=== Scenario 2: Multi-Class Curriculum Results ===")
print(summary_s2)

## 5. Summary & Verification

All Tier-1 multi-seed experimental outputs have been saved to disk under `results/` for figure and table generation in Notebook 05 (`F6`).

In [ ]:
print("Exported files in results/:")
for f in sorted(RESULTS_DIR.glob("tier1_*")):
    print(f"  {f.name} ({f.stat().st_size} bytes)")
print("Tier-1 MCTS-HNM benchmark execution complete.")